# Agent Workflow Management

**Module:** 14 — AI Orchestration

Anatomy of agent workflows: tools, retries, timeouts, cancellation, versioning.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Describe the components of an agent workflow runtime
- Implement retries, timeouts, and cancellation tokens
- Version workflows without breaking in-flight runs


## Anatomy of an Agent Workflow

### Definition
An agent workflow binds a goal, state schema, tool registry, model caller, policy engine, and persistence into a managed run lifecycle.

### Why it matters
Without a clear anatomy, 'agent' means an unmaintainable while-loop.

### How it works
Run = `{run_id, thread_id, state, status, version}`. Steps transition status: `queued→running→waiting_human|succeeded|failed|cancelled`.

### Intuition
A ticket in a job tracker — not a mysterious blob of prompts.

### Pitfalls
- No status model
- Tools unregistered / untyped
- State not schema-validated

### When to use
Any agent you expect to debug in production.


```mermaid
stateDiagram-v2
  [*] --> queued
  queued --> running
  running --> waiting_human
  waiting_human --> running
  running --> succeeded
  running --> failed
  running --> cancelled
```

| Component | Responsibility |
|-----------|----------------|
| Planner / policy | What to do next |
| Tool runtime | Execute side effects |
| Model I/O | Completions / structured outputs |
| State store | Checkpoint & resume |
| Observer | Traces, metrics, logs |


In [ ]:
# Demo 1: run state machine
VALID = {
    ("queued", "running"),
    ("running", "waiting_human"),
    ("waiting_human", "running"),
    ("running", "succeeded"),
    ("running", "failed"),
    ("running", "cancelled"),
}

class Run:
    def __init__(self, run_id):
        self.run_id = run_id
        self.status = "queued"
    def transition(self, to):
        if (self.status, to) not in VALID:
            raise ValueError(f"illegal {self.status}->{to}")
        self.status = to

r = Run("r1")
for s in ["running", "waiting_human", "running", "succeeded"]:
    r.transition(s)
print(r.status)
try:
    r.transition("queued")
except ValueError as e:
    print("blocked", e)


## Retries, Timeouts, Cancellation

### Definition
Operational controls that keep workflows safe under partial failure and user abort.

### Why it matters
Networks flake; models stall; users click Cancel. Ignoring this creates zombie runs and double charges.

### How it works
Classify errors; bound attempts; set deadlines; thread a cancellation token checked between steps.

### Intuition
Kitchen timers + a fire alarm — not optional décor.

### Pitfalls
- Retrying non-idempotent payments
- No cancellation between tool calls
- Eternal waits on humans without SLA

### When to use
All production workflows.


In [ ]:
# Demo 2: timeout + cancellation token
import time

class CancelToken:
    def __init__(self):
        self.cancelled = False
    def cancel(self):
        self.cancelled = True
    def throw_if_cancelled(self):
        if self.cancelled:
            raise RuntimeError("cancelled")

def call_model(token: CancelToken, delay=0.05):
    token.throw_if_cancelled()
    time.sleep(delay)
    token.throw_if_cancelled()
    return {"content": "ok"}

def with_timeout(fn, seconds=0.01):
    # educational: real systems use futures/signals
    start = time.time()
    result = fn()
    if time.time() - start > seconds:
        raise TimeoutError("deadline exceeded")
    return result

tok = CancelToken()
print(call_model(tok))
tok.cancel()
try:
    call_model(tok)
except RuntimeError as e:
    print(e)


In [ ]:
# Demo 3: idempotency key for tool side effects
seen = set()

def charge(idem_key: str, amount: float):
    if idem_key in seen:
        return {"status": "duplicate", "amount": amount}
    seen.add(idem_key)
    return {"status": "charged", "amount": amount}

print(charge("run1:step3", 10))
print(charge("run1:step3", 10))


## Versioning Workflows

### Definition
Pin workflow definition versions so in-flight runs continue on the code they started with, while new runs use new logic.

### Why it matters
Shipping a graph change mid-flight without versioning corrupts state assumptions.

### How it works
Store `workflow_version` on each run; deploy side-by-side handlers; migrate only terminal/queued runs.

### Intuition
Like DB migrations — never rewrite history casually.

### Pitfalls
- Mutating step semantics in place
- No changelog
- State schema mismatch across versions

### When to use
Any workflow you will iterate on more than once (i.e., all of them).


In [ ]:
# Demo 4: side-by-side workflow versions
def workflow_v1(state):
    state["steps"] = state.get("steps", []) + ["classify_v1"]
    return state

def workflow_v2(state):
    state["steps"] = state.get("steps", []) + ["classify_v2_with_langdetect"]
    return state

REGISTRY = {"1.0.0": workflow_v1, "2.0.0": workflow_v2}

def resume(run):
    fn = REGISTRY[run["workflow_version"]]
    return fn(run["state"])

print(resume({"workflow_version": "1.0.0", "state": {}}))
print(resume({"workflow_version": "2.0.0", "state": {}}))


In [ ]:
# Demo 5: tool registry with JSON-schema-ish metadata
TOOLS = {
    "search_kb": {
        "description": "Search knowledge base",
        "parameters": {"q": "string"},
        "idempotent": True,
        "timeout_s": 5,
    },
    "issue_refund": {
        "description": "Refund a charge",
        "parameters": {"order_id": "string", "amount": "number"},
        "idempotent": False,
        "timeout_s": 15,
    },
}

def can_auto_retry(tool_name: str) -> bool:
    return TOOLS[tool_name]["idempotent"]

print({t: can_auto_retry(t) for t in TOOLS})


### Try it yourself — Agent workflows

1. Add `waiting_tool` status to the state machine.
2. Wrap `charge` with retries only when a `Retry-After` simulated error occurs AND key present.
3. Write a migration note from workflow 1.0.0 → 2.0.0 for in-flight `waiting_human` runs.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `run_id` | Unique id for one execution |
| `thread_id` | Conversation/case correlation |
| `cancellation token` | Shared flag to abort cooperatively |
| `side-by-side deploy` | Old+new versions run concurrently |


## Run Lifecycle SLAs

| State | Max time | On breach |
|-------|----------|-----------|
| running model | 60s | cancel + retry or fail |
| running tool | per-tool timeout | mark failed, maybe compensate |
| waiting_human | 24h business | escalate |
| queued | 2m | alert infra |

### Cancellation semantics
- Cooperative: check token between steps  
- Hard: kill worker (may need compensation)  


In [ ]:
# Per-tool timeout registry enforcement
import time

TOOLS = {"search": 0.05, "slow": 0.2}

def exec_tool(name, timeout):
    start = time.time()
    time.sleep(TOOLS[name])
    if time.time() - start > timeout:
        raise TimeoutError(name)
    return "ok"

for n, t in [("search", 0.1), ("slow", 0.1)]:
    try:
        print(n, exec_tool(n, t))
    except TimeoutError as e:
        print("timeout", e)


In [ ]:
# Workflow version pin on run
def start_run(workflow_name, registry):
    return {
        "workflow": workflow_name,
        "workflow_version": registry[workflow_name]["latest"],
        "status": "queued",
    }

reg = {"support": {"latest": "3.2.1", "supported": ["3.1.0", "3.2.1"]}}
print(start_run("support", reg))


### Try it yourself — Workflow mgmt deepen

1. Add `supported` check on resume; fail closed if version removed.
2. Model a kill switch: global `pause_all` prevents queued→running.


## Key Takeaways

- Model statuses explicitly
- Timeouts/cancel/idempotency are core features
- Version workflows like APIs
- Tools need metadata, not just functions
